# 21 · Comparación de pronósticos e incertidumbre del aporte solar
Ejecutar todas las celdas en orden. Se reconstruyen los pronósticos de los cuadernos 17–19, con los mismos horizontes y periodos. La etapa LSTM puede tardar varios minutos. Se exportan las tablas completas como CSV.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/gretadive/correlacion_actividad_solar_elnino/751cd7c187616d8b7ed7b175280dc80a4e999a29/data/processed/dataset_integrado_precipitacion.csv"
archivo = Path("datos.csv")
df = pd.read_csv(archivo if archivo.exists() else url, parse_dates=["fecha"])
df = df.set_index("fecha").sort_index().asfreq("MS")
variables = ["ICEN", "ONI", "precipitacion_mm"]
assert len(df) == 917 and df[variables + ["SSN"]].notna().all().all()
desfases = [0, 3, 6, 12, 24]

horizontes = [3,6,12,24]
resultados = []
print(df.index.min().date(), df.index.max().date(), len(df), "meses")

## Ridge anual
Cada enero se amplía el entrenamiento con objetivos anteriores al inicio del año. Se pronostican los meses siguientes con información hasta t−1 y los mismos horizontes de los cuadernos 14 y 15.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline

# Penalizaciones seleccionadas en los cuadernos 14–15 con validación anterior a 2010.
alfas = {("ICEN",False):[0.1,0.1,0.1,100], ("ICEN",True):[0.1,0.1,100,100],
         ("ONI",False):[0.1,0.1,0.1,10], ("ONI",True):[0.1,0.1,0.1,100]}
filas = []
for indice in ["ICEN","ONI"]:
    for solar in [False,True]:
        variables = [indice,"SSN"] if solar else [indice]
        X = pd.concat([df[v].shift(k).rename(f"{v}_{k}") for v in variables for k in range(1,13)],axis=1)
        for h,alfa in zip(horizontes,alfas[indice,solar]):
            y = df[indice].shift(-h)
            destino = df.index + pd.DateOffset(months=h)
            valido = X.notna().all(axis=1) & y.notna()
            for anio in range(2010,2027):
                corte = pd.Timestamp(anio,1,1)
                prueba = valido & (df.index.year==anio)
                if not prueba.any():
                    continue
                ajuste = valido & (destino < corte)
                modelo = make_pipeline(StandardScaler(),Ridge(alpha=alfa))
                modelo.fit(X[ajuste],y[ajuste])
                for fecha,valor in zip(X[prueba].index,modelo.predict(X[prueba])):
                    filas.append([indice,h,solar,fecha,y.loc[fecha],valor])

parte = pd.DataFrame(filas, columns=["indice","h","solar","origen","observado","pred"])
parte["modelo"] = "Ridge anual"
resultados.append(parte)

## ARIMA/SARIMAX
Comparación retrospectiva con parámetros fijos ajustados hasta diciembre de 2009. ARIMA(2,0,0) y su extensión con SSN usan la misma dinámica, sin componente estacional. El regresor solar se retrasa según el horizonte para evitar utilizar observaciones futuras. La especificación es una referencia inicial.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
filas = []
for indice in ["ICEN","ONI"]:
    serie = df[indice].to_numpy()
    for h in horizontes:
        # El regresor en el mes objetivo s es SSN(s-h-1), conocido al emitir el pronóstico.
        ssn = df.SSN.shift(h+1)
        inicio = h+1
        fin = df.index.get_loc("2010-01-01")
        media, desvio = ssn.iloc[inicio:fin].mean(), ssn.iloc[inicio:fin].std()
        z = ((ssn-media)/desvio).to_numpy()
        for solar in [False,True]:
            exog = z[inicio:fin,None] if solar else None
            modelo = SARIMAX(serie[inicio:fin], exog=exog, order=(2,0,0),trend="c")
            resultado = modelo.fit(disp=False,maxiter=200)
            if not resultado.mle_retvals["converged"]:
                raise RuntimeError("El ajuste requiere revisión")
            par = dict(zip(resultado.param_names,resultado.params))
            beta = par.get("x1",0)
            residuo = serie-beta*z
            for t in range(fin,len(df)-h):
                # AR(2) sobre el índice ajustado por el regresor. Última observación en t-1.
                previo, ultimo = residuo[t-2:t]
                for paso in range(h+1):
                    siguiente = par["intercept"] + par["ar.L1"]*ultimo + par["ar.L2"]*previo
                    previo, ultimo = ultimo, siguiente
                pred = ultimo + beta*z[t+h]
                filas.append([indice,h,solar,df.index[t],serie[t+h],pred])

parte = pd.DataFrame(filas, columns=["indice","h","solar","origen","observado","pred"])
parte["modelo"] = "ARIMA/SARIMAX"
resultados.append(parte)

## LSTM
Redes de ocho unidades con ventanas de 12 meses y horizontes de 3, 6, 12 y 24 meses. La validación 1995–2009 elige la duración de entrenamiento. El ajuste final utiliza objetivos anteriores a 2010. Una sola semilla por experimento. La ejecución completa requiere varios minutos.

In [ ]:
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
filas = []
for indice in ["ICEN","ONI"]:
    for solar in [False,True]:
        variables = [indice,"SSN"] if solar else [indice]
        valores = df[variables].to_numpy()
        indices = np.arange(12,len(df))
        # Ventanas desde t-12 hasta t-1. Una red por horizonte.
        ventanas = np.stack([valores[t-12:t] for t in indices])
        for h in horizontes:
            tf.keras.backend.clear_session()
            tf.keras.utils.set_random_seed(2026)
            validos = indices+h < len(df)
            t = indices[validos]
            X = ventanas[validos]
            y = df[indice].to_numpy()[t+h]
            destino = df.index[t+h]
            train = destino < "1995-01-01"
            val = (df.index[t] >= "1995-01-01") & (destino < "2010-01-01")
            test = df.index[t] >= "2010-01-01"
            escalador = StandardScaler().fit(X[train].reshape(-1,len(variables)))
            X = escalador.transform(X.reshape(-1,len(variables))).reshape(X.shape).astype("float32")
            def red():
                modelo = tf.keras.Sequential([tf.keras.Input((12,len(variables))),
                    tf.keras.layers.LSTM(8),tf.keras.layers.Dense(1)])
                modelo.compile(optimizer="adam",loss="mse")
                return modelo
            def epoca(modelo, mascara):
                posiciones = np.flatnonzero(mascara)
                for i in range(0,len(posiciones),32):
                    lote = posiciones[i:i+32]
                    modelo.train_on_batch(X[lote],y[lote])
            modelo = red()
            errores = []
            for i in range(15):
                epoca(modelo,train)
                pred = modelo(X[val],training=False).numpy().ravel()
                errores.append(np.mean((y[val]-pred)**2))
            mejor_epoca = int(np.argmin(errores))+1
            tf.keras.utils.set_random_seed(2026)
            modelo = red()
            for i in range(mejor_epoca):
                epoca(modelo,destino < "2010-01-01")
            pred = modelo(X[test],training=False).numpy().ravel()
            filas.extend(zip([indice]*len(pred),[h]*len(pred),[solar]*len(pred),df.index[t[test]],y[test],pred))

parte = pd.DataFrame(filas, columns=["indice","h","solar","origen","observado","pred"])
parte["modelo"] = "LSTM"
resultados.append(parte)

## Emparejamiento y referencia de persistencia
Cada modelo se evalúa sobre los mismos meses. Persistencia repite el último índice disponible, correspondiente a t−1.

In [ ]:
predicciones = pd.concat(resultados, ignore_index=True)
clave = ["indice","h","origen"]
assert not predicciones.duplicated(clave+["modelo","solar"]).any()
assert predicciones.groupby(clave).size().eq(6).all()
assert predicciones.groupby(clave).observado.nunique().eq(1).all()
pares = predicciones.pivot(index=clave+["observado","modelo"],columns="solar",values="pred").reset_index()
pares = pares.rename(columns={False:"sin",True:"con"})
def medir(error):
    return np.mean(abs(error)), np.sqrt(np.mean(error**2))
metricas = []
intervalos = []
for (indice,h), grupo in pares.groupby(["indice","h"]):
    ref = grupo[grupo.modelo=="Ridge anual"].sort_values("origen")
    fechas = pd.DatetimeIndex(ref.origen) - pd.DateOffset(months=1)
    error_ref = df.loc[fechas,indice].to_numpy() - ref.observado.to_numpy()
    metricas.append([indice,h,"Persistencia","Referencia",len(ref),*medir(error_ref)])
    for nombre, sub in grupo.groupby("modelo"):
        sub = sub.sort_values("origen")
        assert sub.origen.tolist()==ref.origen.tolist()
        errores = []
        for variante in ["sin","con"]:
            error = (sub[variante]-sub.observado).to_numpy()
            errores.append(error)
            metricas.append([indice,h,nombre,variante,len(error),*medir(error)])
        # Los mismos bloques se extraen para ambos modelos para conservar el emparejamiento.
        base, solar = errores
        for bloque in sorted({h,12,24}):
            rng = np.random.default_rng(2026)
            n = len(sub)
            diferencias = []
            for tanda in range(30):
                inicio = rng.integers(0,n,size=(100,int(np.ceil(n/bloque)),1))
                pos = ((inicio+np.arange(bloque))%n).reshape(100,-1)[:,:n]
                delta_mae = np.mean(abs(solar[pos])-abs(base[pos]),axis=1)
                delta_rmse = np.sqrt(np.mean(solar[pos]**2,axis=1))-np.sqrt(np.mean(base[pos]**2,axis=1))
                diferencias.extend(np.column_stack([delta_mae,delta_rmse]))
            for j,metrica in enumerate(["MAE","RMSE"]):
                li,ls = np.quantile(np.asarray(diferencias)[:,j],[.025,.975])
                intervalos.append([indice,h,nombre,metrica,bloque,medir(solar)[j]-medir(base)[j],li,ls])
tabla = pd.DataFrame(metricas,columns=["indice","h","modelo","variante","n","MAE","RMSE"])
ic = pd.DataFrame(intervalos,columns=["indice","h","modelo","metrica","bloque","delta","IC_inf","IC_sup"])
predicciones.to_csv("21_predicciones.csv",index=False)
tabla.to_csv("21_comparacion.csv",index=False)
ic.to_csv("21_incertidumbre.csv",index=False)
display(tabla.pivot(index=["indice","h"],columns=["modelo","variante"],values="RMSE").round(3))
display(ic.query("bloque==h and metrica=='RMSE'").round(3))

## Interpretación
Delta es el error con SSN menos el error sin SSN. Un valor negativo indica mejora. Un intervalo que contiene cero deja abierta la posibilidad de una diferencia nula. Los intervalos son puntuales y exploratorios, con bootstrap circular de 3000 remuestras. Comparar las longitudes de bloque como análisis de sensibilidad. Los resultados LSTM corresponden a una inicialización. Ridge se reentrena anualmente, mientras que los ajustes ARIMA/SARIMAX y LSTM son fijos durante la prueba.